public0033 Meniscus10 単独64検査診断（提出用ではありません）。
自分のPublic0.950基盤を保持し、異なるDINOv2表現の半月板専門枝の実行可能性を測ります。精度は評価しません。
固定weak manifestのUID先頭64件だけを使い、gold58 UIDと重ならないことを準備時に確認しています。
Notebookはtrain.csv・Report・教師を読みません。実MRIデコードはKaggleに接続された64検査のみです。
competitionとrenta0426/rsna-knee-public0033-meniscus-bag-v1 Version2の計2 Inputsが必要です。
ApexのCoAtNet・ConvNeXt重みは、この単独診断には不要です。
Original Container31481/Python3.13、TeslaT4 x2、Internet OFF。
author stock runtimeはFP32・batch32・10重複窓・6slots・12slices・336pxのまま1回呼びます。
64件のuint8 cacheはscratch disk memmapへ1件ずつ書きます。全件匿名RAM配列は作りません。
実model strict load、12所見有限予測、UID順、slot masks、decoder、時間、RSS、両GPUのpeak VRAMを観測します。
PUBLIC0033_PROFILE_SUMMARY.jsonはすべての契約に成功した場合のみ保存します。失敗時はeventsとtracebackを確認します。
submission.csvは作りません。2列raw CSVと12列diagnostic CSVは採点に提出しないでください。
弱教師全件fitでありOOFモデルではありません。半月板の融合改善や隠しデータ完走は未確認です。
外部checkpoint/runtime Apache2.0。DINOv2とcompetition dataの条件は別に適用されます。
Source: https://www.kaggle.com/datasets/renta0426/rsna-knee-public0033-meniscus-bag-v1/versions/2
Cache reference: https://www.kaggle.com/code/sujanmajhisuzan/rsna-knee-tri-specialist-superstack?scriptVersionId=355742320


In [ ]:
# Code and fixed UID metadata only: no weights, MRI, Reports or teachers.
import base64, hashlib, io, json, sys, zipfile
from pathlib import Path
_profile_payload = base64.b64decode('...')
assert hashlib.sha256(_profile_payload).hexdigest() == '761b5f2c203194856f9435df3b03287f4e5e721fbb0d1f36d7bd3e59ff781ae4'
_profile_members = {'public0033_streaming_adapter.py': {'bytes': 22810, 'sha256': '324b85cadf163820dd4d93381721523396e129a26ffb7952c7711f2496e516d4'}, 'public0033_profile_runtime.py': {'bytes': 26147, 'sha256': '6e06462c651fc3a881be15e60895ca07fd530afce8a743a7fd27b43e22bd69ed'}, 'public0033_profile_cohort.json': {'bytes': 5311, 'sha256': '158883a195a3e54f5ebdfac15d535f99e99ffc8354d76f026a7300c01cbb9997'}}
_profile_source = Path('/kaggle/working/public0033_profile_src')
_profile_source.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(_profile_payload)) as _profile_zip:
    assert set(_profile_zip.namelist()) == set(_profile_members)
    for _name in _profile_zip.namelist():
        assert Path(_name).name == _name and not Path(_name).is_absolute()
        _bytes = _profile_zip.read(_name)
        assert len(_bytes) == _profile_members[_name]['bytes']
        assert hashlib.sha256(_bytes).hexdigest() == _profile_members[_name]['sha256']
        (_profile_source / _name).write_bytes(_bytes)
sys.path.insert(0, str(_profile_source))
print('Prepared code-only profile helpers. No packages are installed or downloaded.')


In [ ]:
from public0033_profile_runtime import run_profile
_cohort = json.loads((_profile_source/'public0033_profile_cohort.json').read_text(encoding='utf-8'))
_summary = run_profile(_cohort, input_root='/kaggle/input',
    output_directory='/kaggle/working/public0033_profile64', scratch='/kaggle/temp')
assert _summary['status'] == 'passed'
assert not Path('/kaggle/working/submission.csv').exists()
